# Word Embeddings — Bag of Words & TF-IDF

**Pipeline:** raw text → sentence tokenize → clean each sentence → build corpus → vectorize (BoW & TF-IDF)

**Key fixes over the practice version**
1. Stopword set is built **once** (not rebuilt on every word) — big speed win.
2. NLTK resources downloaded **once** in a setup cell.
3. Cleaning logic wrapped in a single-responsibility `clean_sentence()` function.
4. Outputs shown as labelled pandas DataFrames (words as columns) — business-readable.

> Note on lemmatization: `WordNetLemmatizer` assumes every word is a **noun** by default,
> so verb forms (e.g. "conquered") are left unchanged. POS-aware lemmatization would fix
> this — kept simple here, flagged for awareness.

## 1. Imports

In [1]:
import re

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# from nltk.stem.porter import PorterStemmer   # stemming alternative (not used here)

## 2. One-time NLTK downloads
Run once per environment. Safe to re-run (skips if already present).

In [2]:
for resource in ("punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"):
    nltk.download(resource, quiet=True)

## 3. The text data

In [3]:
paragraph = """I have three visions for India. In 3000 years of our history, people from all over
the world have come and invaded us, captured our lands, conquered our minds.
From Alexander onwards, the Greeks, the Turks, the Moguls, the Portuguese, the British,
the French, the Dutch, all of them came and looted us, took over what was ours.
Yet we have not done this to any other nation. We have not conquered anyone.
We have not grabbed their land, their culture, their history and tried to enforce
our way of life on them. Why? Because we respect the freedom of others. That is why my
first vision is that of freedom. I believe that India got its first vision of this in 1857,
when we started the War of Independence. It is this freedom that we must protect and nurture
and build on. If we are not free, no one will respect us. My second vision for India is
development. For fifty years we have been a developing nation. It is time we see ourselves as
a developed nation. We are among the top 5 nations of the world in terms of GDP. We have a
10 percent growth rate in most areas. Our poverty levels are falling. Our achievements are
being globally recognised today. Yet we lack the self-confidence to see ourselves as a
developed nation, self-reliant and self-assured. Isn't this incorrect? I have a third vision.
India must stand up to the world. Because I believe that unless India stands up to the world,
no one will respect us. Only strength respects strength. We must be strong not only as a
military power but also as an economic power. Both must go hand-in-hand. My good fortune was
to have worked with three great minds. Dr. Vikram Sarabhai of the Dept. of space, Professor
Satish Dhawan, who succeeded him and Dr. Brahm Prakash, father of nuclear material. I was
lucky to have worked with all three of them closely and consider this the great opportunity
of my life. I see four milestones in my career."""

## 4. Preprocessing setup
Build reusable objects **once** — this is the main efficiency fix.

In [5]:
lemmatizer = WordNetLemmatizer()

STOP_WORDS = set(stopwords.words("english"))


def clean_sentence(sentence: str) -> str:
    """Lowercase, strip non-letters, drop stopwords, lemmatize, re-join."""
    # [^a-zA-Z] = any character that is NOT an English letter -> replace with a space
    text = re.sub("[^a-zA-Z]", " ", sentence)
    text = text.lower()
    words = text.split()                      # split on whitespace -> list of words
    words = [lemmatizer.lemmatize(w) for w in words if w not in STOP_WORDS]
    return " ".join(words)                    # list of words -> single clean string

## 5. Build the corpus
`corpus` = list of cleaned sentence-strings, ready to vectorize.

In [6]:
sentences = nltk.sent_tokenize(paragraph)
corpus = [clean_sentence(s) for s in sentences]

print(f"Total sentences: {len(corpus)}")
corpus[:5]

Total sentences: 32


['three vision india',
 'year history people world come invaded u captured land conquered mind',
 'alexander onwards greek turk mogul portuguese british french dutch came looted u took',
 'yet done nation',
 'conquered anyone']

## 6. Bag of Words (CountVectorizer)
Each cell = **raw count** of a word in a sentence.
`fit` learns the vocabulary → `transform` converts sentences to count rows.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
X_bow = cv.fit_transform(corpus).toarray()   # .toarray(): sparse -> dense grid

print("BoW matrix shape (sentences, vocabulary):", X_bow.shape)

# Business-readable view: words as columns
bow_df = pd.DataFrame(X_bow, columns=cv.get_feature_names_out())
bow_df.head()

BoW matrix shape (sentences, vocabulary): (32, 114)


,achievement,alexander,also,among,anyone,area,assured,believe,brahm,british,...,turk,unless,vikram,vision,war,way,worked,world,year,yet
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,0
2,0,1,0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 7. TF-IDF (TfidfVectorizer)
Same grid shape — but each cell = **TF × IDF**.
Common words (low IDF) get downweighted; rare, distinctive words get lifted.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer()
X_tf = tf.fit_transform(corpus).toarray()

print("TF-IDF matrix shape:", X_tf.shape)

tfidf_df = pd.DataFrame(X_tf, columns=tf.get_feature_names_out()).round(3)
tfidf_df.head()

TF-IDF matrix shape: (32, 114)


,achievement,alexander,also,among,anyone,area,assured,believe,brahm,british,...,turk,unless,vikram,vision,war,way,worked,world,year,yet
0,0.0,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,...,0.000,0.0,0.0,0.549,0.0,0.0,0.0,0.000,0.000,0.000
1,0.0,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,...,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.259,0.305,0.000
2,0.0,0.289,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.289,...,0.289,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.000,0.000
3,0.0,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.000,...,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.000,0.589
4,0.0,0.000,0.0,0.0,0.746,0.0,0.0,0.0,0.0,0.000,...,0.000,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.000,0.000


## 8. Quick sanity peek
Top distinctive words in the first sentence by TF-IDF weight.

In [9]:
first_row = tfidf_df.iloc[0]
first_row[first_row > 0].sort_values(ascending=False).head(8)

three     0.631
india     0.549
vision    0.549
Name: 0, dtype: float64